In [1]:
import pandas as pd
import numpy as np

---

## Level 1 — Student term scores

No hints. Answer these four questions:

- Stack the subject columns into a long format. Which subject has the highest average score overall?
- Which student has the highest average score across all subjects and both terms?
- Unstack the `term` level. For each student, compute their improvement from T1 to T2 in Math. Who improved the most?
- Use `np.corrcoef` to check whether Math and Science scores are correlated (use all 6 rows).

In [58]:
scores = pd.DataFrame({
    'student': ['Alice','Alice','Bob','Bob','Carol','Carol'],
    'term':    ['T1','T2','T1','T2','T1','T2'],
    'Math':    [88, 84, 72, 76, 91, 95],
    'Science': [79, 82, 85, 88, 92, 96],
    'English': [91, 90, 68, 74, 87, 89],
})

# Your code here

l = pd.melt(
    scores,
    value_vars=['Math','Science','English'],
    id_vars = ['student','term'],
    value_name = 'score',
    var_name= 'subject'
)

print(l.groupby('student')['score'].mean().idxmax(),'has the highest average score')

us = l.set_index(['student','term','subject'])
us = us.unstack('term')
us['improv'] = us['score']['T2'] - us['score']['T1']
usm = us.xs('Math',level = 'subject')
print(usm['improv'].idxmax(),'has the most improvment in math')
cors = np.corrcoef(scores['Math'],scores['Science'])[0,1]
print('moderately correlated' if 0.5 < cors < 0.8 else 
      'not very correlated' if cors <= 0.5 else
        'very correlated')

Carol has the highest average score
Bob has the most improvment in math
not very correlated


---

## Level 2 — Job postings

The `salary` column contains ranges like `'$90,000-$120,000'`. No hints.

1. Use `.pipe()` to clean: standardize `title` and `dept` to lowercase, convert `remote` to boolean.
2. Use `.str.extract()` to pull the min and max salary from the range string. Strip `$` and `,`, convert to numeric. Add a `mid_salary` column: `(min + max) / 2`.
3. Use `.apply()` on `title` to add a `level` column: title contains `'senior'`, `'staff'`, or `'lead'` → `'Senior'`; contains `'junior'` → `'Junior'`; else → `'Mid'`.
4. Which level has the highest average `mid_salary`?
5. What is the 90th percentile `mid_salary` across all postings? Use `np.percentile`.

In [90]:
postings = pd.DataFrame({
    'title':  ['Senior Data Scientist','junior software engineer','Lead Product Manager',
               'SENIOR ANALYST','data engineer','Junior UX Designer',
               'Staff Engineer','senior data analyst'],
    'dept':   ['Data','ENGINEERING','Product','Analytics','DATA',
               'Design','ENGINEERING','data'],
    'salary': ['$90,000-$120,000','$65,000-$85,000','$110,000-$145,000',
               '$75,000-$95,000','$80,000-$105,000','$55,000-$75,000',
               '$130,000-$170,000','$85,000-$110,000'],
    'remote': ['Yes','NO','Yes','yes','No','YES','Yes','no'],
})

# Your code here

def stan(df):
    df['title'] = df['title'].str.lower()
    df['dept'] = df['dept'].str.lower()
    return df
def conv(df):
    df['remote'] = df['remote'].str.lower()
    df['remote'] = df['remote'].map({'yes':True, 'no':False})
    return df

clean = postings.copy().pipe(stan).pipe(conv)
clean['s'] = clean['salary'].str.replace('$','',regex = False)
clean['s'] = clean['s'].str.replace(',','',regex = False)

clean['min'] = pd.to_numeric(clean['s'].str.extract(r'(\d+)-')[0])
clean['max'] = pd.to_numeric(clean['s'].str.extract(r'-(\d+)')[0])
clean['mid'] = (clean['min']+clean['max'])/2

clean['title'] = clean['title'].str.lower()
clean['level'] = clean['title'].apply(lambda x: 'Senior' if 'senior|staff|lead' in x else
                                      'Junior' if 'junior' in x else
                                      'Mid')

print(clean.groupby('level')['mid'].mean().idxmax(),'has the highest mean avg salary')
print(np.percentile(clean['mid'],90))

Mid has the highest mean avg salary
134250.0


---

## Level 3 — Customer orders

Two tables. One duplicate in `orders`. Casing issues in both. No steps.

1. Clean and merge.
2. Add a `revenue` column (`amount × qty`). Use named aggregation to summarize by `segment`: total revenue, average order amount, number of orders.
3. Sort by `date`. Add a `running_total` column per customer using `expanding().sum()` on `revenue` (group by customer, then transform).
4. Which city generates the most total revenue?
5. Use `np.corrcoef` to check whether `amount` and `qty` are correlated across all transactions.

In [105]:
customers = pd.DataFrame({
    'cust_id':  ['C01','C02','C03','C04','C05'],
    'name':     ['Alice Park','BOB CHEN','carol lee','Dave Kim','EVE WONG'],
    'segment':  ['PREMIUM','standard','Premium','STANDARD','standard'],
    'city':     ['New York','Chicago','New York','LA','Chicago'],
})

orders = pd.DataFrame({
    'order_id': ['O01','O02','O03','O04','O05','O06','O07','O08','O09','O10','O11','O02'],
    'cust_id':  ['C01','C02','C01','C03','C04','C02','C05','C01','C03','C04','C05','C02'],
    'amount':   [250, 80, 420, 160, 95, 140, 200, 310, 90, 175, 220, 80],
    'qty':      [2, 1, 3, 2, 1, 2, 2, 3, 1, 2, 2, 1],
    'category': ['Electronics','Clothing','Electronics','HOME','clothing',
                 'CLOTHING','Electronics','HOME','Electronics','Clothing','HOME','Clothing'],
    'date':     pd.to_datetime(['2024-01-01','2024-01-15','2024-02-03','2024-02-20',
                                '2024-03-05','2024-03-18','2024-04-02','2024-04-14',
                                '2024-05-01','2024-05-22','2024-06-10','2024-01-15']),
})

# Your code here
customers['name'] = customers['name'].str.lower()
customers['segment'] = customers['segment'].str.lower()
orders['category'] = orders['category'].str.lower()
customers = customers.drop_duplicates().copy()
ordres = orders.drop_duplicates().copy()

m = pd.merge(
    orders, 
    customers, 
    on = 'cust_id'

)

m['revenue'] = m['amount']*m['qty']

g = m.groupby('segment').agg(
    total_rev = ('revenue','sum'),
    avg_amount = ('amount','mean'),
    num_orders = ('order_id','count')

)
print(g)

m = m.sort_values('date')

def f(x):
    return x.expanding().sum()
m['running_total'] = m.groupby('cust_id')['revenue'].transform(f)

print(m.groupby('city')['revenue'].sum().idxmax(),'generates the most total revenue')

cors = np.corrcoef(m['amount'],m['qty'])[0,1]
print('highly correlated' if cors >0.8 else
      'moderatly correlated' if cors>0.5 else
      'other')

          total_rev  avg_amount  num_orders
segment                                    
premium        3100  246.000000           5
standard       1725  141.428571           7
New York generates the most total revenue
highly correlated
